<a href="https://colab.research.google.com/github/mathu2112/FlyRank-ML-Internship-Repo/blob/main/Copy_of_w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mathu2112/FlyRank-ML-Internship-Repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

<br>

# **ML Task Type - Classification**

<br>
This project is a classification problem because the objective is to assign each content item to a predifined class auch as "High performance" or "Low Preformance".The model learns patterns from hisorical content metrics(eg:- search volume,competition,word_count,CTR,impressions) and predicts the performance category for new content.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
#Loading a dataset
import pandas as pd
import numpy as np
data=pd.read_csv("/content/starter.csv")
data.head()


# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

<br>


# **Target Variable**
The variable we are trying to predict in this case is the performance_label. The variable is created on the basis of rule:-

* High Performance = CTR>= median CTR
* Low Performance = CTR< median CTR

This label was created using a defined rule, not directly observed in a dataset.

In [ ]:
# This cell is for CODE (numbers, a query, a check).

#Creating a target variable using CTR
median_ctr=data["ctr"].median()

data["performance_label"]=np.where(
    data["ctr"] >= median_ctr,
    "High",
    "Low"
)

print("Total number of each label")
print(data["performance_label"].value_counts())

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Total number of each label
performance_label
High    15190
Low     14810
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

<br>

# **Success Metric**
The success metric chosen is : Accuracy

Accuracy measures the propotion of correctly classified content items.

* Accuracy = Correct Predictions/ Total Predictions

A reasonable benchmark can be:-

* If Accuracy score >= 80% = Good Model

If the classes become imbalanced, F1-Score can also be used.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
#Prepare FEATURES:
#ID's are dropped because they do not help in prediction
data=data.drop(columns=["content_id","client_id"],errors="ignore")

#Encodeing categorical columns

encoder = LabelEncoder()

for col in data.select_dtypes(include="object").columns:
    if col != "performance_label":
        data[col] = encoder.fit_transform(data[col])

data["performance_label"] = encoder.fit_transform(data["performance_label"])

#split Features and target
#Initial modelling produced unrealistically high accuracy due to target leakage from derived categorical features. After removing features that directly encoded the target variable, the model evaluation was performed using independent predictors.
leak_columns = [
    "competition_level",
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
    "trend_direction"
]

X = data.drop(
    ["performance_label", "ctr"] + leak_columns,
    axis=1
)
y=data["performance_label"]
print(X.shape)
print(y.shape)

#Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#Building classification model USING RANDOM FOREST

rf = RandomForestClassifier(n_estimators=100,random_state=42)
rf.fit(X_train,y_train)

# Make predictions
predictions = rf.predict(X_test)

#Success Metric

accuracy = accuracy_score(y_test,predictions)
print("Accuracy:",accuracy)



# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


(30000, 29)
(30000,)
Accuracy: 0.9893333333333333


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

<br>

**UNIT OF ANALYSIS**

One row represents one content/article record

Each row represents single piece of content with its characteristics ( such as search volume,competition,word count and engagement metrics) and it observed performance label (High/Low based on CTR).



In [ ]:
# This cell is for CODE (numbers, a query, a check).
lane_df = data[[
    "search_volume",
    "competition",
    "word_count",
    "ctr",
    "performance_label"
]]

lane_df.head()
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


,search_volume,competition,word_count,ctr,performance_label
0,10.0,0.67,3221.0,0.76,0
1,90.0,0.01,2481.0,0.05,1
2,0.0,0.00,3515.0,0.09,0
3,10.0,0.00,NaN,0.49,0
4,0.0,0.00,2803.0,0.13,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

<br>

For this project we used classification and used RandomForestClassifier rather than a fixed rule such as using if-else statement because ML models like Random Forest learns patterns from many features simultaneously instead of relying on one rule. as it considers all featuress together it makes it much better at predicting content performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
if data["ctr"].iloc[0] > 0.5:
    print("High Performance")
else:
    print("Low Performance")

#In this case the computer is not learning anything . It is just following the rule that we give

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


High Performance


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.